# Notebook 13 — a miniature end-to-end transport

Source → packet propagation in three dimensions with time → Sobolev interaction → redistribution → escape. Three runs from the same launch: ε\*, $R(T)$ interpolated, the macroatom. Light curves in toy bands, colour evolution, error against complexity.

In [ ]:
import sys, pathlib, time
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[0] / "src"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rtedu
from rtedu import results, DAY, C
from rtedu.visualization import save_fig, OI
from rtedu.atom import five_level_atom
from rtedu.transport import run
from rtedu.matrix import group_of_line, MacroatomRedistribution, build_R, MatrixRedistribution, interpolate_R
from rtedu.redistribution import EpsilonRedistribution
from rtedu.transport3d import ToyEjecta, launch, transport, light_curve
from rtedu.bands import band_fluxes, magnitudes, ORDER
rng = np.random.default_rng(rtedu.SEEDS["ch13"])
atom = five_level_atom(); nm = 1e7 * atom.lam_cm
ej = ToyEjecta(atom, v_max_c=0.2, n0=30.0, t0=2.0 * DAY, T0=4000.0, alpha_T=0.5, heat_index=1.3)
t1 = 6.0 * DAY
n_init, n_heat = 1500, 1500

## The effective models, tabulated in temperature

$R(T)$ on four groups from the radial macroatom transport of chapter 9 at a grid of temperatures; ε* the best scalar at the launch temperature.

In [ ]:
T_grid = [1500.0, 2000.0, 2500.0, 3000.0, 3500.0, 4000.0]           # T falls from 4000 K at 2 d to ~2300 K at 6 d
g4, ng = group_of_line(atom.nu, 10)                                # full resolution: the interpolated R carries the state, not the binning
def R_radial(T_, seed):
    tau_ = atom.line_list(T_, 30.0, 2 * DAY); m = MacroatomRedistribution(atom, tau_)
    run(np.random.default_rng(seed), atom.nu[0] * 1.001, 2000, atom.nu, tau_, 0.2 * C * 2 * DAY, 2 * DAY, m)
    return build_R(m.events, g4, ng)
Rs = {T_: R_radial(T_, rtedu.SEEDS["ch13"] + i) for i, T_ in enumerate(T_grid)}
eps_star = float(results.load("ch09")["eps_star"])                     # chapter 9's best scalar on this atom
print("R(4000 K), 10 groups, first rows:\n", np.round(Rs[4000.0][:3], 2))

In [ ]:
def model_factory(kind):
    """redistribution model at the packet's own time"""
    cache = {}
    def at(t):
        key = (kind, round(t / DAY, 2))
        if key not in cache:
            T_ = ej.T(t); tau_ = ej.tau(t); emis_ = ej.emis(t)
            if kind == "macroatom":
                cache[key] = MacroatomRedistribution(atom, tau_)
            elif kind == "eps":
                cache[key] = EpsilonRedistribution(eps_star, emis_)
            else:
                Rt = interpolate_R(T_, T_grid, [Rs[x] for x in T_grid])
                cache[key] = MatrixRedistribution(Rt, g4, emis_)
        return cache[key]
    return at

x0, d0, nu0, t0 = launch(np.random.default_rng(rtedu.SEEDS["ch13"] + 50), ej, n_init, n_heat, t1)
edges = np.linspace(ej.t0, 7.0 * DAY, 11); tm = 0.5 * (edges[1:] + edges[:-1]) / DAY
runs = {}
for kind in ("macroatom", "eps", "R"):
    t_run = time.time()
    t_esc, nu_esc, n_int = transport(np.random.default_rng(rtedu.SEEDS["ch13"] + 60), ej, x0, d0, nu0, t0, model_factory(kind))
    ok = np.isfinite(t_esc)
    lc = light_curve(t_esc, edges)
    bands = []
    for a_, b_ in zip(edges[:-1], edges[1:]):
        sel = ok & (t_esc >= a_) & (t_esc < b_)
        bands.append(magnitudes(band_fluxes(nu_esc[sel])) if sel.sum() >= 200 else {b: np.nan for b in ORDER})
    runs[kind] = dict(t_esc=t_esc, nu_esc=nu_esc, n_int=n_int, lc=lc, bands=bands, wall=time.time() - t_run, capped=int((~ok).sum()), escaped_by_7d=int((t_esc < 7 * DAY).sum()))
    print(f"{kind:>9s}: {n_int.mean():.2f} interactions per packet, {runs[kind]['capped']} capped, {runs[kind]['wall']:.1f} s")

## Light curves, colours, and the error against complexity

In [ ]:
ref = runs["macroatom"]
def lc_error(r):
    return float(np.abs(r["lc"] - ref["lc"]).sum() / ref["lc"].sum())
def colour(r, a, b):
    return np.array([m[a] - m[b] for m in r["bands"]])
err = {k: lc_error(runs[k]) for k in ("eps", "R")}
def colour_error(r, c):
    d = colour(r, *c.split("-")) - colour(ref, *c.split("-")); d = d[np.isfinite(d)]
    return float(np.mean(np.abs(d))) if d.size else float("nan")
dcol = {k: {c: colour_error(runs[k], c) for c in ("i-J", "J-K")} for k in ("eps", "R")}
n_colour_bins = int(np.isfinite(colour(ref, "J", "K")).sum())
noise_lc = float(np.sqrt(np.maximum(ref["lc"] * np.diff(edges), 1)).sum() / (ref["lc"] * np.diff(edges)).sum())   # relative Poisson noise of the summed curve
print("bolometric L1 error vs macroatom:", {k: round(v, 3) for k, v in err.items()}, "noise", round(noise_lc, 3))
print(f"mean |colour residual| over {n_colour_bins} well-populated bins:", dcol)
# error against complexity: R at N_g = 1, 2, 4, 10 built at the launch state and used throughout (no T interpolation), to isolate the resolution
complexity = {}
for n_g_ in (1, 2, 4, 10):
    g_, ng_ = group_of_line(atom.nu, n_g_)
    m_ = MacroatomRedistribution(atom, atom.line_list(4000.0, 30.0, 2 * DAY)); run(np.random.default_rng(1), atom.nu[0] * 1.001, 2000, atom.nu, atom.line_list(4000.0, 30.0, 2 * DAY), 0.2 * C * 2 * DAY, 2 * DAY, m_)
    Rt = build_R(m_.events, g_, ng_)
    cache = {}
    def at(t, Rt=Rt, g_=g_):
        key = round(t / DAY, 2)
        if key not in cache: cache[key] = MatrixRedistribution(Rt, g_, ej.emis(t))
        return cache[key]
    t_esc, nu_esc, n_int = transport(np.random.default_rng(rtedu.SEEDS["ch13"] + 70), ej, x0, d0, nu0, t0, at)
    ok_ = np.isfinite(t_esc); bands_ = []
    for a_, b_ in zip(edges[:-1], edges[1:]):
        sel = ok_ & (t_esc >= a_) & (t_esc < b_)
        bands_.append(magnitudes(band_fluxes(nu_esc[sel])) if sel.sum() >= 200 else {b: np.nan for b in ORDER})
    rr = dict(lc=light_curve(t_esc, edges), bands=bands_)
    complexity[n_g_] = dict(numbers=int(Rt.size), err_bol=float(np.abs(rr["lc"] - ref["lc"]).sum() / ref["lc"].sum()), err_colour=0.5 * (colour_error(rr, "i-J") + colour_error(rr, "J-K")))
    print(f"R with {n_g_} groups ({Rt.size} numbers): bolometric error {complexity[n_g_]['err_bol']:.3f}, colour error {complexity[n_g_]['err_colour']:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4))
style = {"macroatom": (OI["black"], "macroatom"), "eps": (OI["red"], f"eps* = {eps_star}"), "R": (OI["blue"], "R(T), 10 groups, interpolated")}
for k, r in runs.items():
    axes[0].plot(tm, r["lc"] * DAY / (n_init + n_heat), "o-", color=style[k][0], label=style[k][1], ms=3)
axes[0].set_xlabel("escape time [d]"); axes[0].set_ylabel("escaped energy per day / total"); axes[0].legend(fontsize=7); axes[0].set_title("bolometric light curve", fontsize=9)
for k, r in runs.items():
    axes[1].plot(tm, colour(r, "i", "J"), "o-", color=style[k][0], ms=3, label=style[k][1] + " (i-J)")
    axes[1].plot(tm, colour(r, "J", "K"), "s--", color=style[k][0], ms=3, alpha=0.6, label=style[k][1] + " (J-K)")
axes[1].set_xlabel("escape time [d]"); axes[1].set_ylabel("colour [mag]"); axes[1].legend(fontsize=6, ncol=2); axes[1].set_title("colour evolution", fontsize=9)
axes[2].plot([complexity[n]["numbers"] for n in complexity], [complexity[n]["err_colour"] for n in complexity], "o-", color=OI["blue"], label="R at N_g = 1, 2, 4, 10: colour")
axes[2].plot([complexity[n]["numbers"] for n in complexity], [complexity[n]["err_bol"] for n in complexity], "s--", color=OI["sky"], label="R at N_g = 1, 2, 4, 10: bolometric")
axes[2].axhline(0.5 * (dcol["eps"]["i-J"] + dcol["eps"]["J-K"]), color=OI["red"], ls="--", label="eps*: colour"); axes[2].axhline(noise_lc, color="grey", ls=":", label="bolometric Poisson noise")
axes[2].set_xscale("log"); axes[2].set_xlabel("numbers stored"); axes[2].set_ylabel("error vs macroatom (mag for colour)"); axes[2].legend(fontsize=6); axes[2].set_title("error against complexity", fontsize=9)
fig.tight_layout(); save_fig(fig, "ch13_lightcurve")

In [ ]:
results.record("ch13", dict(n_init=n_init, n_heat=n_heat, t0_d=ej.t0 / DAY, t1_d=t1 / DAY, T0=ej.T0, alpha_T=ej.alpha_T, T_end=ej.T(t1), v_max_c=0.2, eps_star=eps_star, T_grid=T_grid,
                            n_int={k: float(r["n_int"].mean()) for k, r in runs.items()}, capped={k: r["capped"] for k, r in runs.items()},
                            err=err, dcol=dcol, n_colour_bins=n_colour_bins, noise_lc=noise_lc, complexity={str(k): v for k, v in complexity.items()},
                            t_peak_d={k: float(tm[int(np.argmax(r["lc"]))]) for k, r in runs.items()},
                            lc={k: (r["lc"] * DAY / (n_init + n_heat)).tolist() for k, r in runs.items()}, tm_d=tm))